# Notebook 1 — Bronze → Silver
**Tech Challenge Fase 3 · Big Data to Analytics · POSTECH**

Pipeline de ingestão e limpeza dos dados da pesquisa State of Data Brasil (2021, 2022, 2023).

**Etapas:**
1. Leitura dos CSVs brutos da camada Bronze (S3)
2. Limpeza dos nomes das colunas (remoção de caracteres especiais)
3. Adição da coluna `ano_pesquisa`
4. União dos 3 anos com `unionByName` (preservando colunas faltantes)
5. Checkpoint para quebrar o lineage do Spark
6. Gravação na camada Silver como Parquet

## 1. Inicialização do contexto Glue e Spark

In [ ]:
import sys
import re
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

BUCKET = 's3://tc-fase3-state-of-data-wl'
print('Contexto inicializado com sucesso!')

## 2. Função de limpeza de nomes de colunas

Remove caracteres inválidos para Parquet/Athena (parênteses, barras, espaços, etc.)

In [ ]:
def limpar(nome):
    """Remove caracteres inválidos dos nomes das colunas."""
    nome = re.sub(r'[^a-zA-Z0-9_]', '_', nome)
    nome = re.sub(r'_+', '_', nome)
    return nome.strip('_')

## 3. Leitura dos CSVs Bronze

InferSchema desativado para evitar conflitos de tipo entre os anos.

In [ ]:
df_2021 = spark.read.option('header','true').option('inferSchema','false') \
               .csv(f'{BUCKET}/bronze/State_of_data_2021.csv')
df_2022 = spark.read.option('header','true').option('inferSchema','false') \
               .csv(f'{BUCKET}/bronze/State_of_data_2022.csv')
df_2023 = spark.read.option('header','true').option('inferSchema','false') \
               .csv(f'{BUCKET}/bronze/State_of_data_2023.csv')

print(f'2021: {df_2021.count()} registros, {len(df_2021.columns)} colunas')
print(f'2022: {df_2022.count()} registros, {len(df_2022.columns)} colunas')
print(f'2023: {df_2023.count()} registros, {len(df_2023.columns)} colunas')

## 4. Limpeza das colunas e adição do ano

Usamos `toDF()` para renomear todas as colunas de uma vez, evitando StackOverflow por encadeamento de `withColumnRenamed`.

In [ ]:
df_2021 = df_2021.withColumn('ano_pesquisa', F.lit('2021'))
df_2022 = df_2022.withColumn('ano_pesquisa', F.lit('2022'))
df_2023 = df_2023.withColumn('ano_pesquisa', F.lit('2023'))

# Renomeia todas as colunas de uma vez (evita StackOverflowError)
df_2021 = df_2021.toDF(*[limpar(c) for c in df_2021.columns])
df_2022 = df_2022.toDF(*[limpar(c) for c in df_2022.columns])
df_2023 = df_2023.toDF(*[limpar(c) for c in df_2023.columns])

print('Colunas limpas com sucesso!')
print(f'Exemplo de colunas 2023: {df_2023.columns[:5]}')

## 5. União dos datasets (2021 + 2022 + 2023)

`allowMissingColumns=True` preserva colunas que existem em apenas alguns anos.

In [ ]:
df_silver = df_2021.unionByName(df_2022, allowMissingColumns=True) \
                   .unionByName(df_2023, allowMissingColumns=True) \
                   .dropDuplicates()

# Checkpoint para quebrar o lineage do Spark e evitar StackOverflow
spark.sparkContext.setCheckpointDir(f'{BUCKET}/checkpoints/')
df_silver = df_silver.checkpoint()

total = df_silver.count()
print(f'Total de registros Silver: {total}')
print(f'Total de colunas: {len(df_silver.columns)}')

## 6. Gravação na camada Silver (Parquet)

In [ ]:
df_silver.write.mode('overwrite') \
         .parquet(f'{BUCKET}/silver/state_of_data_consolidado/')

print('Silver gravado com sucesso!')
print(f'Caminho: {BUCKET}/silver/state_of_data_consolidado/')
job.commit()